# 04 · Reinforcement learning with verifiable rewards (GRPO-style)

Simplified from tinker-cookbook's `recipes/rl_loop.py`.

The big change: **the tokens now come from the model itself.** Each step:
1. **Sample** G attempts per problem from the current policy
2. **Score** each one (is the `\boxed{}` answer correct, within 256 tokens? 1 : 0)
3. **Advantage** = reward − mean reward of its group (better than siblings → reinforce)
4. **Train** with `importance_sampling` loss: like SFT, but each sampled token is
   weighted by its advantage (and can be negative!)

In [ ]:
from common import *

train = load_gsm8k("train").shuffle(seed=1)
BATCH_SIZE, GROUP_SIZE, STEPS, LR = 16, 8, 20, 4e-5  # 16 problems x 8 attempts = 128 rollouts/step
sampling_params = types.SamplingParams(max_tokens=MAX_TOKENS, temperature=1.0, stop=renderer.get_stop_sequences())

# Start from the SFT checkpoint if we have one (it already knows the format), else from base.
START_FROM = load_checkpoint("sft")
print("starting from:", START_FROM or f"base {MODEL}")

In [ ]:
# ── LIVE: the reward is just a Python function ────────────────────────────
def reward_fn(response: str, answer: str) -> float:
    return gsm8k_reward(response, answer)  # 1.0 if \boxed{} matches, else 0.0

In [ ]:
# ── LIVE: the RL loop ──────────────────────────────────────────────────────
training_client = new_training_client(START_FROM)

for step in range(STEPS):
    rows = train.select(range(step * BATCH_SIZE, (step + 1) * BATCH_SIZE))

    # 1. Sample from the *current* policy (fresh weights every step = on-policy)
    policy = training_client.save_weights_and_get_sampling_client().result()
    prompts = [renderer.build_generation_prompt(gsm8k_convo(q)) for q in rows["question"]]
    futures = [policy.sample(p, num_samples=GROUP_SIZE, sampling_params=sampling_params) for p in prompts]

    batch, step_rewards = [], []
    for prompt, fut, answer in zip(prompts, futures, rows["answer"]):
        seqs = fut.result().sequences
        # 2. Score each attempt
        texts = [renderers.get_text_content(renderer.parse_response(s.tokens)[0]) for s in seqs]
        rewards = [reward_fn(t, answer) for t in texts]
        step_rewards += rewards
        # 3. Group-relative advantage
        mean_r = sum(rewards) / len(rewards)
        if all(r == mean_r for r in rewards):
            continue  # all right or all wrong -> no signal
        # 4. One datum per attempt; the sampler's logprobs enable the importance-sampling correction
        for s, r in zip(seqs, rewards):
            batch.append(make_rl_datum(prompt, s.tokens, s.logprobs, advantages=r - mean_r))

    if batch:
        training_client.forward_backward(batch, loss_fn="importance_sampling")
        training_client.optim_step(adam(LR)).result()
    print(f"step {step:2d}  mean reward = {sum(step_rewards) / len(step_rewards):.3f}  ({len(batch)} datums)")

In [ ]:
sampling_client = training_client.save_weights_and_get_sampling_client().result()
eval_gsm8k(sampling_client, label="4. RL" + (" (from SFT)" if START_FROM else ""))
save_checkpoint(training_client, "rl")

**Takeaway:** compare to SFT. SFT taught *every token* of a reference answer.
RL gives one bit per attempt (right/wrong) smeared over hundreds of tokens: a very
sparse signal, but it learns from the model's *own* mistakes.

**Bridge to distillation:** what if we could keep the on-policy sampling, but
get a grade on *every token* instead of one per answer?